In [1]:
print('hello')

hello


In [2]:
import os 
os.chdir('../')

In [3]:
from langchain.document_loaders import PyPDFLoader, DirectoryLoader
from langchain.text_splitter import RecursiveCharacterTextSplitter

c:\football-analytics-rag\footy-guide\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [4]:
import os

os.chdir("C:/football-analytics-rag")

print(os.getcwd())
# Extracting text from PDF files in the 'data' directory
def load_pdfs_from_directory(data):
    loader = DirectoryLoader(data, 
                             glob="*.pdf", 
                             loader_cls=PyPDFLoader
                             )
    documents = loader.load()
    return documents

C:\football-analytics-rag


In [5]:
extracted_documents = load_pdfs_from_directory('data')

In [6]:
extracted_documents

[Document(metadata={'producer': 'pikepdf 8.15.1', 'creator': 'arXiv GenPDF (tex2pdf:)', 'creationdate': '', 'author': 'Gabriel Anzer; Kilian Arnsmeyer; Pascal Bauer; Joris Bekkers; Ulf Brefeld; Jesse Davis; Nicolas Evans; Matthias Kempe; Samuel J Robertson; Joshua Wyatt Smith; Jan Van Haaren', 'doi': 'https://doi.org/10.48550/arXiv.2505.15820', 'license': 'http://creativecommons.org/licenses/by/4.0/', 'ptex.fullbanner': 'This is pdfTeX, Version 3.141592653-2.6-1.40.25 (TeX Live 2023) kpathsea version 6.3.5', 'title': 'Common Data Format (CDF): A Standardized Format for Match-Data in Football (Soccer)', 'trapped': '/False', 'arxivid': 'https://arxiv.org/abs/2505.15820v4', 'source': 'data\\Common Data Format.pdf', 'total_pages': 31, 'page': 0, 'page_label': '1'}, page_content='The Common Data Format (CDF):\nA Standardized Format for Match-Data in Football (Soccer)\nGabriel Anzer1, Kilian Arnsmeyer2, Pascal Bauer2,3, Joris Bekkers4, 5, 13, Ulf Brefeld6, Jesse Davis7,\nNicolas Evans8, Matt

In [7]:
len(extracted_documents)

1568

In [8]:
from typing import List
from langchain.schema import Document

In [9]:
def filter_to_minimal_docs(docs: List[Document]) -> List[Document]:
    """
    Given a list of Document objects, return a new list of Document objects
    containing only 'source' in metadata and the original page_content.
    """
    min_docs: List[Document] = []
    for doc in docs:
        src = doc.metadata.get("source")
        min_docs.append(
            Document(
                page_content=doc.page_content,
                metadata={"source": src}
            )
        )
    return min_docs

In [10]:
min_docs = filter_to_minimal_docs(extracted_documents)

In [11]:
min_docs

[Document(metadata={'source': 'data\\Common Data Format.pdf'}, page_content='The Common Data Format (CDF):\nA Standardized Format for Match-Data in Football (Soccer)\nGabriel Anzer1, Kilian Arnsmeyer2, Pascal Bauer2,3, Joris Bekkers4, 5, 13, Ulf Brefeld6, Jesse Davis7,\nNicolas Evans8, Matthias Kempe9, Samuel J Robertson 8, Joshua Wyatt Smith10, 11, and Jan Van\nHaaren7, 12\n1RB Leipzig, Leipzig, Germany\n2Deutscher Fußball-Bund (DFB), Frankfurt, Germany\n3Saarland University, Saarbr¨ ucken, Germany\n4U.S. Soccer Federation, Chicago, USA\n5UnravelSports, Breda, Netherlands\n6Leuphana University, L¨ uneburg, Germany\n7KU Leuven, Leuven.AI, & LISS, Heverlee, Belgium\n8FIFA, Zurich, Switzerland\n9University of Groningen, Groningen, Netherlands\n10Wyatt AI Inc., Montreal, Canada\n11Concordia University, Montreal, Canada\n12Club Brugge, Brugge, Belgium\n13PySport, Eindhoven, Netherlands\nDecember 2024\nAbstract\nDuring football matches, a variety of different parties (e.g., companies) each 

In [12]:
# Splitting the documents into smaller chunks for better processing
def text_split(min_docs):
    text_splitter = RecursiveCharacterTextSplitter(
        chunk_size=500,
        chunk_overlap=20,
    )
    texts_chunk = text_splitter.split_documents(min_docs)
    return texts_chunk

In [13]:
texts_chunk = text_split(min_docs)
print(f"Number of text chunks: {len(texts_chunk)}")

Number of text chunks: 8050


In [14]:
texts_chunk

[Document(metadata={'source': 'data\\Common Data Format.pdf'}, page_content='The Common Data Format (CDF):\nA Standardized Format for Match-Data in Football (Soccer)\nGabriel Anzer1, Kilian Arnsmeyer2, Pascal Bauer2,3, Joris Bekkers4, 5, 13, Ulf Brefeld6, Jesse Davis7,\nNicolas Evans8, Matthias Kempe9, Samuel J Robertson 8, Joshua Wyatt Smith10, 11, and Jan Van\nHaaren7, 12\n1RB Leipzig, Leipzig, Germany\n2Deutscher Fußball-Bund (DFB), Frankfurt, Germany\n3Saarland University, Saarbr¨ ucken, Germany\n4U.S. Soccer Federation, Chicago, USA\n5UnravelSports, Breda, Netherlands'),
 Document(metadata={'source': 'data\\Common Data Format.pdf'}, page_content='6Leuphana University, L¨ uneburg, Germany\n7KU Leuven, Leuven.AI, & LISS, Heverlee, Belgium\n8FIFA, Zurich, Switzerland\n9University of Groningen, Groningen, Netherlands\n10Wyatt AI Inc., Montreal, Canada\n11Concordia University, Montreal, Canada\n12Club Brugge, Brugge, Belgium\n13PySport, Eindhoven, Netherlands\nDecember 2024\nAbstract\n

In [15]:
from langchain_huggingface import HuggingFaceEmbeddings

def download_embeddings():
    """
    Download and return the HuggingFace embeddings model.
    """
    model_name = "sentence-transformers/all-MiniLM-L6-v2"
    embeddings = HuggingFaceEmbeddings(
        model_name=model_name
    )
    return embeddings

embedding = download_embeddings()

In [16]:
embedding

HuggingFaceEmbeddings(model_name='sentence-transformers/all-MiniLM-L6-v2', cache_folder=None, model_kwargs={}, encode_kwargs={}, query_encode_kwargs={}, multi_process=False, show_progress=False)

In [17]:
vector = embedding.embed_query("Hello world")
vector

[-0.03447720408439636,
 0.031023239716887474,
 0.00673496862873435,
 0.026108969002962112,
 -0.03936196118593216,
 -0.16030246019363403,
 0.06692393124103546,
 -0.0064414795488119125,
 -0.047450557351112366,
 0.014758911915123463,
 0.0708753690123558,
 0.05552756413817406,
 0.01919337548315525,
 -0.026251327246427536,
 -0.010109500028192997,
 -0.026940541341900826,
 0.022307470440864563,
 -0.02222665585577488,
 -0.14969269931316376,
 -0.01749308407306671,
 0.007676247972995043,
 0.054352279752492905,
 0.003254473675042391,
 0.03172597661614418,
 -0.0846213549375534,
 -0.0294059906154871,
 0.051595624536275864,
 0.048124030232429504,
 -0.003314792178571224,
 -0.05827920511364937,
 0.04196930304169655,
 0.022210685536265373,
 0.1281888484954834,
 -0.02233896590769291,
 -0.011656301096081734,
 0.06292833387851715,
 -0.032876282930374146,
 -0.09122605621814728,
 -0.031175389885902405,
 0.05269956216216087,
 0.0470348559319973,
 -0.08420302718877792,
 -0.03005620837211609,
 -0.0207447819411

In [18]:
print(f"Length of the embedding vector: {len(vector)}")

Length of the embedding vector: 384


In [19]:
from dotenv import load_dotenv
import os
load_dotenv()

True

In [20]:
PINECONE_API_KEY = os.getenv("PINECONE_API_KEY")
GOOGLE_API_KEY = os.getenv("GOOGLE_API_KEY")


os.environ["PINECONE_API_KEY"] = PINECONE_API_KEY
os.environ["GOOGLE_API_KEY"] = GOOGLE_API_KEY

In [21]:
from pinecone import Pinecone 
pinecone_api_key = PINECONE_API_KEY

pc = Pinecone(api_key=pinecone_api_key)

In [22]:
pc

In [24]:
from pinecone import ServerlessSpec 

index_name = "footy-bot"

if not pc.has_index(index_name):
    pc.create_index(
        name = index_name,
        dimension=384,  # Dimension of the embeddings
        metric= "cosine",  # Cosine similarity
        spec=ServerlessSpec(cloud="aws", region="us-east-1")
    )


index = pc.Index(index_name)

In [25]:
from langchain_pinecone import PineconeVectorStore

docsearch = PineconeVectorStore.from_documents(
    documents=texts_chunk,
    embedding=embedding,
    index_name=index_name
)

In [26]:
# Load Existing index 

from langchain_pinecone import PineconeVectorStore
# Embed each chunk and upsert the embeddings into your Pinecone index.
docsearch = PineconeVectorStore.from_existing_index(
    index_name=index_name,
    embedding=embedding
)

# Add more data to the existing Pinecone index

In [28]:
tisini_football = Document(
    page_content="""
    Tisini Football Definitions is a football analytics knowledge base that explains
    key football data concepts, event definitions, and performance metrics used in
    match analysis.

    The football section is organized into six main areas: Break In Play, Goalkeeping,
    Advanced Passing, Shots and Attempts, Advanced Metrics, and supporting notes. It
    covers 33 terms including:

    - Break In Play events: goal kicks (short/long, complete/incomplete), fouls
      (foul won/conceded), throw-ins (long/normal), free kicks (won/taker), cards
      (yellow, second yellow, straight red), offside, and corners (short, in-swinging,
      out-swinging).
    - Goal actions: goals by type (open play, set-piece, penalty-rebound, counter,
      transition, direct free kick) and penalties (won, conceded, miss).
    - Goalkeeping: saves (normal/penalty), throw-outs, kick-outs, claims (catch,
      punch, miss, drop), and run-outs (successful/unsuccessful).
    - Advanced passing: assists, progressive passes (complete/incomplete), and
      crosses (left/right; complete/incomplete/blocked).
    - Shots and attempts: shots (in-box/out-box), shots on target, off target,
      and blocked shots.
    - Advanced metrics: key passes, set-piece chances (corner/free kick/throw-in),
      interceptions (own half/opponent half), box carries, box touches (receiving,
      dribbling, passing in box), blocks, clearances, tackles (successful/lost),
      dribbles (complete/incomplete/multiple-player), ball events (ball won/lost),
      and aerial duels (won/foul/multiple-player).

    These globally consistent definitions help analysts, coaches, scouts, and football
    data enthusiasts understand how football events are recorded, interpreted, and
    used for tactical and performance analysis.
    """,
    metadata={
        "source": "Tisini Football Definitions",
        "category": "Football Analytics",
        "url": "https://tisini-definitions.vercel.app/"
    }
)

In [29]:
docsearch.add_documents(documents=[tisini_football])

['a9fc339b-06ac-450f-a4b4-ac815291e481']

In [30]:
retriever = docsearch.as_retriever(search_type="similarity", search_kwargs={"k":3})

In [33]:
retrieved_docs = retriever.invoke("What is a goal kick?")
retrieved_docs

[Document(id='c98f9abb-f4c3-4721-8b8f-5a6d34909494', metadata={'source': 'data\\Football Rules Simplified (IFAB).pdf'}, page_content='What is a goal kick in football?\nA goal kick is the restart of play after the whole of the ball has gone out over the goal line (but not into the goal), on the ground or in the air, and \nwas last touched by an attacking-team player.\nA goal kick is taken by the defending team from anywhere in the goal area.\nWhat should happen at a goal kick in football?\nThe ball must be stationary (not moving).'),
 Document(id='81a5314c-7773-4178-b111-11a31e8cd27a', metadata={'source': 'data\\Laws of the Game 2025_26.pdf'}, page_content='it is in play,  the goal kick is retaken.\nThe Goal Kick'),
 Document(id='37a37a5e-00bb-4cdc-b388-9b234085c1e3', metadata={'source': 'data\\Laws of the Game 2025_26.pdf'}, page_content='•\u2002a corner kick if it enters the team’s goal')]

In [48]:
from langchain_groq import ChatGroq

chatModel = ChatGroq(
    model="llama-3.3-70b-versatile",
    temperature=0.2
)

In [49]:
from langchain.chains import create_retrieval_chain
from langchain.chains.combine_documents import create_stuff_documents_chain
from langchain_core.prompts import ChatPromptTemplate

In [54]:
system_prompt = """
You are a Football Analytics Assistant.

Answer questions using only the provided football knowledge context.
Do not invent definitions, statistics, or information. If the answer is not in
the context, say you don't have enough information.

When explaining football concepts:
- Give a clear definition.
- Explain how it is recorded in football data.
- Explain its relevance in match analysis when useful.
- Use simple examples when helpful.

Maintain an accurate and professional tone suitable for football analysts,
coaches, scouts, and data enthusiasts.

Context:
{context}
"""
prompt = ChatPromptTemplate.from_messages(
    [
        ("system", system_prompt),
        ("human", "{input}"),
    ]
)

In [55]:
question_answer_chain = create_stuff_documents_chain(chatModel, prompt)
rag_chain = create_retrieval_chain(retriever, question_answer_chain)

In [60]:
response = rag_chain.invoke(
    {
        "input": "The difference between a manager and a coach?"
    }
)

print(response["answer"])

In the context of football, the terms "manager" and "coach" are often used interchangeably, but they have distinct roles and responsibilities.

A **coach** is an individual employed by a club to perform specific football-related tasks, such as:

* Training and coaching players
* Selecting players for matches and competitions
* Making tactical choices during matches and competitions

A coach's primary focus is on the technical and tactical aspects of the game, and they are responsible for the development and implementation of the team's playing style and strategy.

On the other hand, a **manager** is responsible for the overall management of the team, including administrative and organizational tasks, such as:

* Managing the team's budget and finances
* Overseeing recruitment and scouting
* Handling media relations and communications
* Making key decisions about the team's direction and strategy

In some cases, a single person may combine the roles of manager and coach, but this requir

In [62]:
response = rag_chain.invoke(
    {
        "input": "What is a formation in football?"
    }
)

print(response["answer"])

In football, a formation refers to the strategic arrangement of players on the pitch, excluding the goalkeeper. It describes how the players are positioned and organized in terms of their roles and responsibilities, both defensively and offensively.

In football data, formations are typically recorded using a numerical notation, such as 4-4-2, 3-3-3, or 3-5-2. This notation represents the number of players in each line of the formation, starting from the defense and moving forward to the attack. For example, a 4-4-2 formation indicates that the team has four defenders, four midfielders, and two forwards.

The relevance of formations in match analysis lies in their impact on a team's playing style, strengths, and weaknesses. By analyzing a team's formation, coaches and analysts can identify areas of vulnerability, anticipate their opponent's strategy, and develop effective counter-measures. Formations can also influence the team's ability to maintain possession, create scoring opportuni

In [67]:
response = rag_chain.invoke(
    {
        "input": "What are shots in football?"
    }
)

print(response["answer"])

In football, a shot refers to an attempt by a player to score a goal by kicking, heading, or using another part of their body to direct the ball towards the opponent's goal. 

In football data, shots are recorded and categorized based on the body part used to attempt the shot, such as feet, head, or other body parts. The context provides specific codes for different types of shots, including:
- Backheel (89): a shot taken with the heel
- Diving Header (90): a shot attempted with a header while diving
- Half Volley (91): contact made off the ground and after a bounce
- Lob (92): a shot with a high arc trajectory to pass over an opposition player
- Normal (93): a shot that does not fall into any other technique

Shots are a crucial aspect of match analysis, as they can indicate a team's attacking intent, creativity, and goal-scoring ability. Analyzing shots can help coaches and analysts identify areas of strength and weakness in a team's attack, as well as inform tactical decisions to im

In [70]:
response = rag_chain.invoke(
    {
        "input": "What is a tackle in football?"
    }
)

print(response["answer"])

In football, a tackle is defined as when a player connects with the ball in a legal, ground-level challenge. This typically occurs when a defensive player attempts to win the ball back from an opponent who is in possession.

In terms of recording tackles in football data, a tackle is usually awarded when a defensive player successfully challenges an opponent for the ball, either by sliding into the path of the ball or moving into the direction of the player in possession with the aim of removing possession.

The relevance of tackles in match analysis lies in their ability to disrupt the opponent's attack and gain possession of the ball. A high number of successful tackles by a team or player can indicate strong defensive skills and a ability to win the ball back quickly.

For example, if a defender slides into the path of the ball and wins it back from an opponent who is attempting to dribble past them, this would be recorded as a tackle. Conversely, if the defender misses the ball or 